# Evaluation of Agent Systems

## Learning goals

- understand why agent workflows need structured evaluation instead of anecdotal demos
- inspect the balanced 40-question evaluation dataset
- compute and interpret correctness, retrieval, grounding, abstention, latency, and reasoning-step metrics
- compare the baseline and agent workflow with tables and charts


## Concept explanation

Evaluation is only meaningful if the notebook is running in the expected environment. This cell prints the active interpreter and the current runtime profile so you can quickly sanity-check local versus DGX execution.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import RuntimeConfig

print(sys.executable)
print(RuntimeConfig.auto_detect())

## Why evaluation matters

A single impressive answer does not tell us whether the architecture is reliable. Evaluation turns design claims into measured evidence. In this repository, the baseline workflow is useful because it gives us a strong control condition for every later improvement.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.evaluator import load_eval_dataset, run_evaluation_suite

dataset = load_eval_dataset()
dataset_frame = pd.DataFrame(dataset)
print(f'Total questions: {len(dataset_frame)}')
dataset_frame[['id', 'question_type', 'expected_status', 'question']].head(12)

## Designing evaluation datasets

A good evaluation set is not just large enough. It should also be balanced enough that failures are interpretable. The expanded dataset now holds eight questions per query type, which makes comparisons across lookup, comparison, summary, multi-hop, and abstention cases much easier to discuss.


## Implementation

The evaluation pipeline is implemented in `src/evaluator.py`. We will first inspect the dataset distribution, then run repeated baseline-versus-agent comparisons, and finally visualize the trade-offs.

In [ ]:
distribution = dataset_frame['question_type'].value_counts().sort_index()
ax = distribution.plot(kind='bar', color='#4C78A8', title='Evaluation Dataset Distribution by Query Type')
ax.set_xlabel('query_type')
ax.set_ylabel('question count')
plt.tight_layout()
plt.show()
distribution.reset_index().rename(columns={'index': 'question_type', 'question_type': 'count'})

## Metrics

The built-in evaluation pipeline repeatedly runs both the baseline and the agent workflow, then computes answer correctness, retrieval hit rate, grounding pass rate, abstain precision, latency, and average reasoning steps. We persist outputs so later notebooks can reuse them.


In [ ]:
results, summary = run_evaluation_suite(repeats=2, persist_outputs=True)
summary

## Compare baseline vs agent

Tables are useful, but they are easier to interpret when broken down by query type. The next cell shows both the system-level summary and a per-type comparison so you can see where the agent workflow helps most.


In [ ]:
question_type_breakdown = (
    results.groupby(['system', 'expected_question_type'])[[
        'answer_correctness',
        'retrieval_hit_rate',
        'grounding_pass_rate',
        'abstain_precision',
        'latency_seconds',
        'average_steps',
    ]]
    .mean()
    .round(3)
)

display(summary)
display(question_type_breakdown)

## Experiment

A radar chart is a helpful teaching device because it shows multi-metric trade-offs in a single picture. Here we compare answer correctness, retrieval hit rate, grounding pass rate, abstain precision, and a latency-friendly inverse latency score.


In [ ]:
radar = summary.set_index('system')[['answer_correctness', 'retrieval_hit_rate', 'grounding_pass_rate', 'abstain_precision']].copy()
radar['speed_score'] = 1.0 / summary.set_index('system')['latency'].clip(lower=0.001)
radar['speed_score'] = radar['speed_score'] / radar['speed_score'].max()
radar_metrics = list(radar.columns)
angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={'projection': 'polar'})
for system, row in radar.iterrows():
    values = row.tolist()
    values += values[:1]
    ax.plot(angles, values, label=system)
    ax.fill(angles, values, alpha=0.15)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_metrics)
ax.set_title('Baseline vs Agent Workflow Radar View')
ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1.1))
plt.tight_layout()
plt.show()
radar.round(3)

## Result analysis

The central trade-off to watch is that the agent workflow usually spends more steps and sometimes more latency to gain better grounding and abstention behavior. That is a worthwhile trade in a research or internal-assistant setting where trustworthiness matters.


In [ ]:
metric_deltas = summary.set_index('system').loc['agent_workflow'] - summary.set_index('system').loc['baseline']
metric_deltas.to_frame(name='agent_minus_baseline').round(3)

## Takeaways

- Evaluation turns architecture ideas into measurable claims.
- Balanced datasets matter because they make failure patterns easier to interpret.
- Stronger grounding and abstention often cost more steps than the baseline.
- The next notebook uses these results to ask not just *what failed*, but *why* it failed.
